# Point Cloud to CAD-sequence

In this notebook the complete interactive pipeline for encoding point clouds into a latent space, from which DeepCAD decodes a CAD-sequence.

In [1]:
import os
import sys
import shutil
import glob
import json
import argparse
import importlib

import numpy as np
import pandas as pd
pd.set_option('display.max_rows', None)
import h5py
import torch
import torch.nn.functional as F

from OCC.Core.BRepCheck import BRepCheck_Analyzer
from OCC.Extend.DataExchange import read_step_file, write_step_file
from OCC.Core.STEPControl import STEPControl_Reader
from OCC.Core.StlAPI import StlAPI_Writer
from OCC.Core.BRepMesh import BRepMesh_IncrementalMesh

sys.path.append("..")
sys.path.append("../code")

from dataset import PointCloudEmbeddingSequenceDataset, PointCloudEmbeddingDataset
from models.DeepCAD.config.configAE import ConfigAE
from models.DeepCAD.trainer.trainerAE import TrainerAE
from models.DeepCAD.cadlib.extrude import CADSequence
from models.DeepCAD.cadlib.visualize import vec2CADsolid, create_CAD
from models.DeepCAD.utils.file_utils import ensure_dir
from models.DeepCAD.cadlib.macro import ALL_COMMANDS, CMD_ARGS_MASK, EOS_IDX, SOL_IDX, EXT_IDX, ARC_IDX

### PointNet++

In [2]:
def inplace_relu(m):
    classname = m.__class__.__name__
    if classname.find('ReLU') != -1:
        m.inplace=True

def load_pointnet():
    
    saved_model = torch.load(model_path, map_location=torch.device('cpu'), weights_only=True)
    state_dict = saved_model['model_state_dict']
    config = saved_model['config']
    if 'module.' in next(iter(state_dict)):
        monitor.log_and_print("Model was saved wrapped in nn.DataParallel.\nRemoving 'module.' from state dict.")
        state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}

    sys.path.append(os.path.join('..', 'models','Pointnet_Pointnet2_pytorch', 'models'))
    
    model = importlib.import_module(config['model_type'])
    if 'architecture' in config:
        if config['architecture'] == 'own':
            classifier = model.get_model(256, normal_channel=False)
        elif config['architecture'] == "copy_author":
            classifier = model.get_model_copy_author(256, normal_channel=False)
        elif config['architecture'] == "tanh":
            classifier = model.get_model_tanh(256, normal_channel=False)
    else:
        classifier = model.get_model(256, normal_channel=False)
    criterion = model.get_loss_mse()
    classifier.apply(inplace_relu) 
    
    
    classifier.load_state_dict(state_dict)
    classifier.eval()
    print(f"Loading PointNet++ from {os.path.abspath(model_path)}")
    return classifier

### DeepCAD

In [3]:
def load_deepcad(cfg):
    tr_agent = TrainerAE(cfg)
    tr_agent.load_ckpt(cfg.ckpt)
    tr_agent.net.eval()
    return tr_agent

### Load data

In [4]:
def get_data(indices, dataset):
    pc_list = []
    lat_rep_list = []  
    pc_paths = []
    cad_seq_list = []
    pc_dir = os.path.join(results_dir, "infered_point_clouds")
    if os.path.exists(pc_dir):
        shutil.rmtree(pc_dir)
    os.mkdir(pc_dir)
    
    for i in indices:
        data = dataset[i]
        pc, cad_seq, lat_rep, _ = data["pc"], data["tgt_vec"], data["z"], data["id"]
        pc_path = os.path.abspath(dataset.get_pc_path(i))
        pc_path_destination = os.path.join(pc_dir, os.path.basename(pc_path))
        shutil.copy2(pc_path, pc_path_destination)
        pc_paths.append(pc_path_destination)
        pc_list.append(pc)
        lat_rep_list.append(lat_rep)
        cad_seq_list.append(cad_seq)

    with h5py.File(h5_file, 'a') as hf:
        dt = h5py.special_dtype(vlen=str)
        path_dataset = hf.create_dataset("pc_paths", shape=(len(pc_paths),), dtype=dt)
        path_dataset[:] = pc_paths
        
    pc_batch = torch.stack(pc_list, dim=0)
    lat_rep_batch = torch.stack(lat_rep_list, dim=0)
    cad_seq_batch = torch.stack(cad_seq_list, dim=0)
    
    return pc_batch, lat_rep_batch, cad_seq_batch

### Inference

In [5]:
def infer_pointnet(indices, dataset, model):
    with h5py.File(h5_file, 'w') as hf:
        z_pred = hf.create_dataset('z_pred', 
                                   shape=(len(indices), latent_dim), 
                                   dtype=np.float32)
        z_target = hf.create_dataset('z_target',
                                     shape=(len(indices), latent_dim),
                                     dtype = np.float32)
        seq_target = hf.create_dataset('seq_target', 
                                       shape=(len(indices), cfg.max_total_len, cfg.n_args + 1), 
                                       dtype=np.int64)
        
        pc, lat_rep, cad_seq = get_data(indices, dataset)
        z_target[:] = lat_rep
        seq_target[:] = cad_seq

        criterion_loader = importlib.import_module('pointnet2_cls_ssg')
        criterion = criterion_loader.get_loss_mse()
        
        with torch.no_grad():
            pc = pc.transpose(2, 1)
            pred, _ = model(pc)
            z_pred[:] = pred.detach()
            loss = criterion(pred, lat_rep)
            print(f"Avg. MSE-Loss: {loss.detach().item():.8f}")
            return pred, cad_seq

In [6]:
def infer_deepcad(pred, cad_seq, tr_agent):
    with h5py.File(h5_file, 'a') as hf:
        seq_pred = hf.create_dataset('seq_pred', 
                                     shape=(pred.shape[0], cfg.max_total_len, cfg.n_args + 1), 
                                     dtype=np.int64)
        cmd_logits = hf.create_dataset('cmd_logits', 
                                       shape=(pred.shape[0], cfg.max_total_len, cfg.n_commands), 
                                       dtype=np.float32)
        args_logits = hf.create_dataset('args_logits', 
                                       shape=(pred.shape[0], cfg.max_total_len, cfg.n_args, cfg.args_dim + 1), 
                                       dtype=np.float32)
        with torch.no_grad():
            pred = pred.unsqueeze(dim = 1)
            output = tr_agent.decode(pred)

            output["tgt_commands"] = cad_seq[:, :, 0] 
            output["tgt_args"] = cad_seq[:, :, 1:]
            loss_dict = tr_agent.loss_func(output)

            cmd_logits[:] = output['command_logits']
            args_logits[:] = output['args_logits']
            
            batch_out_vec = tr_agent.logits2vec(output)
            
            seq_pred[:] = batch_out_vec
            
            print(f"Avg. Command-Loss: {loss_dict['loss_cmd'].detach().cpu().item():.8f}")
            print(f"Avg. Argument-Loss: {loss_dict['loss_args'].detach().cpu().item():.8f}")

            return {"tgt_commands": cad_seq[:, :, 0],  "tgt_args": cad_seq[:, :, 1:], "pred": batch_out_vec}


### Utils

In [7]:
def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=-1, keepdims=True)

In [8]:
def cross_entropy(logits, target):
    logits = torch.from_numpy(logits).unsqueeze(0)
    target = torch.tensor([target]).long()
    #print(logits.shape, target.shape)
    return F.cross_entropy(logits, target)

In [67]:
def find_index(file_id, ds):
    """To find specific files.
    If provided with a file id (str, no extension), returns the index in the dataset.
    """
    pcs = ds.pc
    
    for i, data in enumerate(pcs):
        id = os.path.splitext(os.path.basename(data))[0]
        if id == file_id:
            print(i)
            break

### Visualization

In [10]:
def show_results(idx):
    with h5py.File(h5_file, "r") as hf:
        pc_path = hf['pc_paths'][idx].decode("utf-8")
        args_logits = hf['args_logits'][idx]
        cmd_logits = hf['cmd_logits'][idx]
        seq_pred = hf['seq_pred'][idx]
        seq_target = hf['seq_target'][idx]
        z_pred = hf['z_pred'][idx]
        z_target = hf['z_target'][idx]

    print(f"Point Cloud path: {pc_path}")
    for idx, command in enumerate(ALL_COMMANDS):
        print(f"{idx} -> {command}")

    target_commands = []
    predicted_commands = []
    pred_commands_prob = []
    cmd_loss = []
    cmd_loss_torch = []
    target_commands_prob = []
    
    all_pred_commands = list(seq_pred[:, 0])
    seq_length = list(seq_target[:, 0]).index(EOS_IDX) + 3
    cmd_logits_softmax = softmax(cmd_logits[:seq_length, :])

    for i in range(seq_length):
        predicted_commands.append(int(all_pred_commands[i]))
        target_commands.append(int(seq_target[i, 0]))
        pred_commands_prob.append(round(float(cmd_logits_softmax[i, predicted_commands[i]]) * 100, 5))
        cmd_loss.append(cross_entropy(cmd_logits[i,:], target_commands[i]).item())
        target_commands_prob.append(round(float(cmd_logits_softmax[i, target_commands[i]]) * 100, 5))
    
    df = pd.DataFrame(list(zip(target_commands, predicted_commands, target_commands_prob, pred_commands_prob, cmd_loss)),
                      columns=['trgt', 'pred','prob_trgt', 'prob_pred', 'loss'])
    print(f"Sum CADLoss for {seq_length} commands:  {sum(cmd_loss):.8f}")
    print(f"Mean CADLoss for {seq_length} commands: {np.mean(cmd_loss):.8f}")
    return df

In [11]:
def show_results_args(sample_idx, cmd_idx):
    with h5py.File(h5_file, "r") as hf:
        pc_path = hf['pc_paths'][sample_idx].decode("utf-8")
        args_logits = hf['args_logits'][sample_idx]
        cmd_logits = hf['cmd_logits'][sample_idx]
        seq_pred = hf['seq_pred'][sample_idx]
        seq_target = hf['seq_target'][sample_idx]
        z_pred = hf['z_pred'][sample_idx]
        z_target = hf['z_target'][sample_idx]

    print(f"Point Cloud path: {pc_path}")

    # Extract predicted and target commands
    seq_length = list(seq_target[:, 0]).index(EOS_IDX) + 3
    predicted_commands = seq_pred[:seq_length, 0]      # (60)
    target_commands = seq_target[:seq_length, 0]       # (60)

    # Create lists of predicted and target commands for each of the 16 arguments of each command
    predicted_command_list = [cmd for cmd in predicted_commands[:seq_length] for _ in range(cfg.n_args)] # (16 * seq_length)
    target_command_list = [cmd for cmd in target_commands[:seq_length] for _ in range(cfg.n_args)]       # (16 * seq_length)

    # Extract the arguments logits and calculate softmax
    args_logits = args_logits[:seq_length]        # (seq_length, 16, 257)
    args_softmax = softmax(args_logits)           # (seq_length, 16, 257)

    # Extract the target arguments and softmax probabilities
    target_args = list(seq_target[:seq_length, 1:])                                                           # (seq_length, 16)
    target_cmd_args = torch.tensor([trgt_arg for trgt_cmd_args in target_args for trgt_arg in trgt_cmd_args]) # (16 * seq_length)
    target_cmd_args_sm = args_softmax[torch.repeat_interleave(torch.arange(seq_length), cfg.n_args), torch.arange(cfg.n_args).repeat(seq_length), target_cmd_args + 1] # (16 * seq_length)
    
    # Extract the predicted arguments and softmax probabilities
    predicted_cmd_args = []
    for i, cmd_args in enumerate(args_logits):
        pred_args = np.argmax(cmd_args, -1)
        pred_args = [x - 1 for x in pred_args]
        predicted_cmd_args += list(pred_args)   # (16 * seq_length) 
    predicted_cmd_args_sm = args_softmax[torch.repeat_interleave(torch.arange(seq_length), cfg.n_args), torch.arange(cfg.n_args).repeat(seq_length), [x + 1 for x in predicted_cmd_args]] # (16 * seq_length)
    
    # Calculate cross-entropy loss
    arg_loss = []
    counter = 0
    for i, cmd_args in enumerate(args_logits):
        for j, arg_logits in enumerate(cmd_args):
            loss = cross_entropy(arg_logits, target_cmd_args[counter] + 1)
            arg_loss.append(loss.item())
            counter += 1

    # Extract data for single command view
    if cmd_idx != -1:
        target_command_list = target_command_list[cfg.n_args * cmd_idx:(cfg.n_args * cmd_idx) + cfg.n_args]
        predicted_command_list = predicted_command_list[cfg.n_args * cmd_idx:(cfg.n_args * cmd_idx) + cfg.n_args]
        target_cmd_args = target_cmd_args[cfg.n_args * cmd_idx:(cfg.n_args * cmd_idx) + cfg.n_args]
        predicted_cmd_args = predicted_cmd_args[cfg.n_args * cmd_idx:(cfg.n_args * cmd_idx) + cfg.n_args]
        target_cmd_args_sm = target_cmd_args_sm[cfg.n_args * cmd_idx:(cfg.n_args * cmd_idx) + cfg.n_args]
        predicted_cmd_args_sm = predicted_cmd_args_sm[cfg.n_args * cmd_idx:(cfg.n_args * cmd_idx) + cfg.n_args]
        arg_loss = arg_loss[cfg.n_args * cmd_idx:(cfg.n_args * cmd_idx) + cfg.n_args]

    # Filter out all arguments not involved in the final loss (marked by -1 using a mask in the original code)
    filtered_lists = [[x for x, trgt_arg in zip(lst, target_cmd_args) if trgt_arg != -1] 
                      for lst in [target_command_list, 
                                  predicted_command_list, 
                                  target_cmd_args, 
                                  predicted_cmd_args, 
                                  target_cmd_args_sm, 
                                  predicted_cmd_args_sm, 
                                  arg_loss]]
    target_command_list, predicted_command_list, target_cmd_args, predicted_cmd_args, target_cmd_args_sm, predicted_cmd_args_sm, arg_loss = filtered_lists

    # Create dataframe
    df = pd.DataFrame(list(zip(target_command_list, 
                               predicted_command_list, 
                               [x.item()  for x in target_cmd_args],
                               [x.item()  for x in predicted_cmd_args],
                               [round(x * 100, 5)  for x in target_cmd_args_sm],
                               [round(x * 100, 5) for x in predicted_cmd_args_sm], 
                               [round(x, 5) for x in arg_loss])),
                      columns=['trgt cmd', 'pred cmd', 'trgt', 'pred','prob_trgt', 'prob_pred', 'loss'])
    
    print(f"Mean args loss multiplied by {cfg.loss_weights['loss_args_weight']}: {cfg.loss_weights['loss_args_weight'] * np.mean(arg_loss):.8f}")
    return df

### Export to .step, .stl and .obj

In [12]:
def export2step():
    form = "h5"
    filter = True
    output_dir = os.path.join(results_dir, "step_files")
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.mkdir(output_dir)
    h5_path = os.path.join(results_dir, "data.h5")

    with h5py.File(h5_path, 'r') as fp:
        out_vec = fp['seq_pred'][:].astype(np.float64)
        names = fp['pc_paths'][:]
        for i, seq in enumerate(out_vec):
            pc_path = names[i].decode('utf-8')
            out_shape = vec2CADsolid(seq)
    
            if filter:
                analyzer = BRepCheck_Analyzer(out_shape)
                if not analyzer.IsValid():
                    print(f"CAD-sequence of {os.path.basename(pc_path)} is invalid.")
                    continue
    
            pc_name = os.path.splitext(os.path.basename(pc_path))[0]
            save_path = os.path.join(output_dir, pc_name + ".step")
            write_step_file(out_shape, save_path)


In [13]:
def step2stl():
    step_files = glob.glob(os.path.join(results_dir, "step_files", "*.step"))
    obj_dir = os.path.join(results_dir, "stl_files")
    if os.path.exists(obj_dir):
        shutil.rmtree(obj_dir)
    os.mkdir(obj_dir)

    for step_file in step_files:

        save_path = os.path.join(obj_dir, os.path.splitext(os.path.basename(step_file))[0] + ".stl")
    
        step_reader = STEPControl_Reader()
        step_reader.ReadFile(step_file)
        step_reader.TransferRoots()
        shape = step_reader.OneShape()

        BRepMesh_IncrementalMesh(shape, 0.5)

        stl_writer = StlAPI_Writer()
        stl_writer.Write(shape, save_path)

In [14]:
def step2obj(): # .obj files from this function lead to malformed file error in cloud compare
    step_files = glob.glob(os.path.join(results_dir, "step_files", "*.step"))
    obj_dir = os.path.join(results_dir, "obj_files")
    if os.path.exists(obj_dir):
        shutil.rmtree(obj_dir)
    os.mkdir(obj_dir)
    
    for step_file in step_files:
        shape = read_step_file(step_file)
        save_path = os.path.join(obj_dir, os.path.splitext(os.path.basename(step_file))[0] + ".obj")
        write_step_file(shape, save_path)

### Metrics

In [15]:
def calculate_ACC(results):

    TOLERANCE = 3

    # overall accuracy
    avg_cmd_acc = [] # ACC_cmd
    avg_param_acc = [] # ACC_param
    
    # accuracy w.r.t. each command type
    each_cmd_cnt = np.zeros((len(ALL_COMMANDS),))
    each_cmd_acc = np.zeros((len(ALL_COMMANDS),))

    # accuracy w.r.t each parameter
    args_mask = CMD_ARGS_MASK.astype(np.float32)
    N_ARGS = args_mask.shape[1]
    each_param_cnt = np.zeros([*args_mask.shape])
    each_param_acc = np.zeros([*args_mask.shape])

    B = results["tgt_commands"].shape[0]

    for i in range(B): # for each sample in the batch
        seq_length = list(results["tgt_commands"][i]).index(EOS_IDX)
        out_cmd = results["pred"][i,:seq_length,0]
        gt_cmd = results["tgt_commands"][i, :seq_length].numpy()
    
        out_param = results["pred"][i,:seq_length,1:]
        gt_param = results["tgt_args"][i, :seq_length].numpy()

        print("Commands")
        print("out ", out_cmd)
        print("gt  ", gt_cmd)

        cmd_acc = (out_cmd == gt_cmd).astype(np.int32)
        print("acc ", cmd_acc)
        param_acc = []
              
        for j in range(len(gt_cmd)):
            cmd = gt_cmd[j]
            if not cmd == 4:
                print("\nCMD: ", cmd)
            else:
                print("\nCMD: ", cmd, " not considered")
            each_cmd_cnt[cmd] += 1
            each_cmd_acc[cmd] += cmd_acc[j]
            if cmd in [SOL_IDX, EOS_IDX]:
                continue
            print("Count of commands:               ", each_cmd_cnt)
            print("Count of correct pred. commands: ", each_cmd_acc)
        
            if out_cmd[j] == gt_cmd[j]: # NOTE: only account param acc for correct cmd
                tole_acc = (np.abs(out_param[j] - gt_param[j]) < TOLERANCE).astype(np.int32)
                print("Correct predicted args: ", tole_acc.tolist())
                
                # filter param that do not need tolerance (i.e. requires strictly equal)
                if cmd == EXT_IDX:
                    tole_acc[-2:] = (out_param[j] == gt_param[j]).astype(np.int32)[-2:]
                elif cmd == ARC_IDX:
                    tole_acc[3] = (out_param[j] == gt_param[j]).astype(np.int32)[3]

                print("mask:                   ", args_mask[cmd])
                valid_param_acc = tole_acc[args_mask[cmd].astype(bool)].tolist()
                print("Valid params per cmd:   ", valid_param_acc)
                param_acc.extend(valid_param_acc)
                print("param_acc ", param_acc)
    
                each_param_cnt[cmd, np.arange(N_ARGS)] += 1
                print("Each param cnt\n", each_param_cnt)
                each_param_acc[cmd, np.arange(N_ARGS)] += tole_acc
                print("Each param acc\n", each_param_cnt)

        print("\nBool predicted paramas: ", param_acc)
        if len(param_acc) == 0: # No cmd was correct, therefore no param is recorded
            param_acc = 0       # Therefore the param accuarcy for this sample is 0
        else:
            param_acc = np.mean(param_acc)
        
        avg_param_acc.append(param_acc)
        cmd_acc = np.mean(cmd_acc)
        avg_cmd_acc.append(cmd_acc)

        print("Sample Avg cmd acc: ", avg_cmd_acc[i])
        print("Sample Avg arg acc: ", avg_param_acc[i])

    print("LLOOOL", avg_cmd_acc)
    avg_cmd_acc = np.mean(avg_cmd_acc)
    print("\n\navg command acc (ACC_cmd):", avg_cmd_acc)
    
    avg_param_acc = np.mean(avg_param_acc)
    
    print("avg param acc (ACC_param):", avg_param_acc)

    # acc of each command type
    print("Correct cmd count: ", each_cmd_acc)
    each_cmd_acc = each_cmd_acc / (each_cmd_cnt + 1e-6)
    print("each command count:", each_cmd_cnt)
    print("each command acc:", each_cmd_acc)

    # acc of each parameter type
    each_param_acc = each_param_acc * args_mask
    each_param_cnt = each_param_cnt * args_mask
    each_param_acc = each_param_acc / (each_param_cnt + 1e-6)

    for i in range(each_param_acc.shape[0]):
        print(ALL_COMMANDS[i] + " param acc:", each_param_acc[i][args_mask[i].astype(bool)])

## Start

### Variables

Store the models in ```experiments```, a results directory will be created for each respective model.

In [32]:
model_name = "non_aug_best"

In [33]:
model_path = os.path.join("experiments", model_name) + ".pth"
results_dir = os.path.join("experiments", model_name + "_results")
if not os.path.exists(results_dir):
    os.mkdir(results_dir)
h5_file = os.path.join(results_dir, "data.h5")
cfg = ConfigAE('test', model_path="../data/latent", parse=False)
latent_dim = 256

In [34]:
pointnet_plusplus = load_pointnet()
deepcad = load_deepcad(cfg)

Loading PointNet++ from /Users/saidharb/Documents/LocalDocuments/Master-Thesis/Point-Cloud-Reconstruction/notebooks/experiments/non_aug_best.pth
Loading checkpoint from /Users/saidharb/Documents/LocalDocuments/Master-Thesis/Point-Cloud-Reconstruction/data/latent/pretrained/model/ckpt_epoch1000.pth ...


In [38]:
dataset = PointCloudEmbeddingSequenceDataset("../data", 'train')
print(f"Dataset contains {len(dataset)} samples.")

Dataset contains 160982 samples.


In [40]:
find_index("00637471")

4146


In [42]:
indices = [4561]

In [43]:
pred, trgt_cad_seq = infer_pointnet(indices, dataset, pointnet_plusplus)

Avg. MSE-Loss: 0.10910706


In [44]:
result = infer_deepcad(pred, trgt_cad_seq, deepcad)

Avg. Command-Loss: 2.11831188
Avg. Argument-Loss: 13.25144100


In [45]:
show_results(0)

Point Cloud path: experiments/non_aug_best_results/infered_point_clouds/00716729.ply
0 -> Line
1 -> Arc
2 -> Circle
3 -> EOS
4 -> SOL
5 -> Ext
Sum CADLoss for 9 commands:  19.06480916
Mean CADLoss for 9 commands: 2.11831213


,trgt,pred,prob_trgt,prob_pred,loss
0,4,4,99.99923,99.99923,0.000008
1,0,0,99.98835,99.98835,0.000117
2,1,0,0.01639,99.98361,8.716410
3,0,0,99.97461,99.97461,0.000254
4,1,0,0.00321,99.99679,10.348022
5,5,5,100.00000,100.00000,0.000000
6,3,3,100.00000,100.00000,0.000000
7,3,3,100.00000,100.00000,0.000000
8,3,3,100.00000,100.00000,0.000000


In [46]:
show_results_args(0,-1) 

Point Cloud path: experiments/non_aug_best_results/infered_point_clouds/00716729.ply
Mean args loss multiplied by 2.0: 13.25144037


,trgt cmd,pred cmd,trgt,pred,prob_trgt,prob_pred,loss
0,0,0,140,154,0.01981,28.08540,8.52662
1,0,0,116,83,0.01257,26.26030,8.98153
2,1,0,223,223,99.97932,99.97932,0.00021
3,1,0,150,128,0.00000,65.39973,17.30749
4,1,0,32,72,0.95919,27.74695,4.64684
5,1,0,0,0,97.30949,97.30949,0.02727
6,0,0,223,223,76.51369,76.51369,0.26770
7,0,0,167,128,0.00000,99.84226,17.15433
8,1,0,128,128,100.00000,100.00000,0.00000
9,1,0,128,128,100.00000,100.00000,0.00000


### Accuracy Calculation

In order to calculate the command and argument accuracy:

**Commands**
- mark each correct predicted command with a 1 in a (60) tensor and all false with 0
- Count how many times each command is in gt ```each_cmd_cnt```
- Count how many times each command was correctly predicted ```each_cmd_acc``` (max: number of times indicated in ```each_cmd_cnt```)
- For final average cmd acc:
    - per sample: Average the tensor indicating which command was predicted correctly
    - per batch: Average per sample metrics
- per command metrics:
    - Divide number of each correct predicted command by total number of each command in gt
 
**Args**
- For each correct predicted command count the number of correct predicted params within tolerance (tolerance is not employe for absolute values like the counter-clockwise flag in Arcs or some extrusion parameters)
- Mask out not needed params for each command
- Create a list which indicates for each argument for all correctly predicted commands if the argument is predicted correctly (or within tolerance)
- For final average param acc:
    - per sample: Average the tensor indicating which arg was predicted correctly
    - per batch: Average per sample metrics
- per arg metrics:
    - mask out all not used args per command
    - Divide number of each correct predicted arg by total number of each arg in gt

In [26]:
calculate_ACC(result)

Commands
out  [4 0 0 0 0 5]
gt   [4 0 1 0 1 5]
acc  [1 1 0 1 0 1]

CMD:  4  not considered

CMD:  0
Count of commands:                [1. 0. 0. 0. 1. 0.]
Count of correct pred. commands:  [1. 0. 0. 0. 1. 0.]
Correct predicted args:  [0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
mask:                    [1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Valid params per cmd:    [0, 0]
param_acc  [0, 0]
Each param cnt
 [[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]
Each param acc
 [[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

In [27]:
export2step()


*******************************************************************
******        Statistics on Transfer (Write)                 ******

*******************************************************************
******        Transfer Mode = 0  I.E.  As Is       ******
******        Transferring Shape, ShapeType = 2                      ******
** WorkSession : Sending all data
 Step File Name : experiments/copy_author_best_results/step_files/00716729.step(380 ents)  Write  Done


In [28]:
step2stl()

## TODO: 
- export2step and step2stl also for target CAD sequence
- multiple sample visualization for commands and args
- Include quantization to the visualization in the future

### Gedanken

Done: 
- had to finish applications
- first thing I did was refactor training
    - Automatic resume of training if cluster fails
    - Parallelization (30mins/epoch -> 12 mins/epoch)
- implemented test script
- Implemented cosine annealing learning rate -> show new training with better convergence!
- Worked on CAD Loss understanding
- Implemented testing pipeline to make sure the data is alligned
- Finished pc2cad pipeline with thorough understanding of loss for train/val/test
- Created interactive pc2cad
- cmd loss visualization
- args loss visualization
- Calculation of ACC_cmd and ACC_param

HiWi:
- created SAiL poster
- documented literature research
- made Blensor work

Next:

- train DeepCAD ourselves?
- Train both models in one pipeline using CADLoss?
- use blensor to create new data?


In [2]:
import os

In [30]:
lol = "haha/dcbb/lol.ply"
print(os.path.splitext(os.path.basename(lol))[0])

lol


In [31]:
import h5py
file = "/Users/saidharb/Documents/LocalDocuments/Master-Thesis/special_samples/00025814_vec.h5"
with h5py.File(file, 'r') as hf:
    print(hf.keys())
    print(np.asarray(hf['out_vec']))
    print(np.asarray(hf['gt_vec']))

FileNotFoundError: [Errno 2] Unable to open file (unable to open file: name = '/Users/saidharb/Documents/LocalDocuments/Master-Thesis/special_samples/00025814_vec.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
lol = PointCloudEmbeddingSequenceDataset("../data", "test")
print(len(lol))

In [ ]:
from torch.utils.data import DataLoader
dataloader = DataLoader(lol, batch_size = 192, num_workers = 0, shuffle = False)

In [ ]:
for i in dataloader:
    print(i["pc"].shape)
    break

In [1]:
lol2 = PointCloudEmbeddingDataset("../data", "test")

NameError: name 'PointCloudEmbeddingDataset' is not defined

In [ ]:
dataloader2 = DataLoader(lol2, batch_size = 192, num_workers = 0, shuffle = False)

In [ ]:
for i,j in dataloader2:
    print(i.shape)
    break

## Calculate normals - important change to dataset.py!!

__Disable__ the sampling of 2048 points in ```dataset.py```. The normals of all 8096 points should be calculated.

In [2]:
import open3d as o3d

In [11]:
train = PointCloudEmbeddingSequenceDataset("../data", "train")
val = PointCloudEmbeddingSequenceDataset("../data", "validation")
test = PointCloudEmbeddingSequenceDataset("../data", "test")

In [5]:
corrupt_files = []
counter = 0
import time
for dataset in [train, val, test]:
    start = time.time()
    print("START")
    for i, data in enumerate(dataset):
        print(i, end='\r')
        try:
            data = dataset[i]
            pc_path = dataset.get_pc_path(i)
            new_pc_path = pc_path.replace('pc_cad','pc_cad_norm')
            os.makedirs(os.path.dirname(new_pc_path), exist_ok=True)
            
            point_cloud = o3d.io.read_point_cloud(pc_path)
            point_cloud.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamKNN(knn=50))
            point_cloud.orient_normals_consistent_tangent_plane(100)
            
            normals = np.asarray(point_cloud.normals)
            points = np.asarray(point_cloud.points)
            pc_n = np.hstack((points, normals))
            
            pcn = o3d.geometry.PointCloud()
            pcn.points = o3d.utility.Vector3dVector(points)
            pcn.normals = o3d.utility.Vector3dVector(normals)

            o3d.io.write_point_cloud(new_pc_path, pcn)
        
        except Exception as e:
            counter += 1
            corrupt_files.append(pc_path)
            
    duration = (time.time() - start)/60
    print(f"{round(duration, 2)} minutes END\n")
    
end_duration = (time.time() - start)/60
print(f"{round(end_duration, 2)} minutes FINNISH\n")
print(counter, corrupt_files)

START
2483.88 minutes END

START
134.12 minutes END

START
121.11 minutes END

121.11 minutes FINNISH

12 ['../data/pc_cad/0063/00637471.ply', '../data/pc_cad/0074/00745094.ply', '../data/pc_cad/0064/00642362.ply', '../data/pc_cad/0042/00426282.ply', '../data/pc_cad/0061/00616862.ply', '../data/pc_cad/0064/00642363.ply', '../data/pc_cad/0054/00547409.ply', '../data/pc_cad/0057/00577907.ply', '../data/pc_cad/0045/00453085.ply', '../data/pc_cad/0009/00090870.ply', '../data/pc_cad/0005/00056341.ply', '../data/pc_cad/0083/00830386.ply']


In [148]:
pc_path = '../data/pc_cad/0083/00830386.ply'
new_pc_path = pc_path.replace('pc_cad','pc_cad_norm')

point_cloud = o3d.io.read_point_cloud(pc_path)
point_cloud.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamKNN(knn=50))
point_cloud.orient_normals_consistent_tangent_plane(100)

normals = np.asarray(point_cloud.normals)
points = np.asarray(point_cloud.points)
pc_n = np.hstack((points, normals))
norms = np.linalg.norm(pc_n[:, 3:], axis=1)
print(np.unique(norms))

pcn = o3d.geometry.PointCloud()
pcn.points = o3d.utility.Vector3dVector(points)
pcn.normals = o3d.utility.Vector3dVector(normals)

o3d.io.write_point_cloud(new_pc_path, pcn)

RuntimeError: QH6013 qhull input error: input is less than 4-dimensional since all points have the same x coordinate 0.266

While executing:  | qhull d Qbb Qt
Options selected for Qhull 2020.2.r 2020/08/31:
  run-id 1188969886  delaunay  Qbbound-last  Qtriangulate  _pre-merge
  _zero-centrum  Pgood  _max-width 0.71  Error-roundoff 7.5e-16
  _one-merge 6.7e-15  _near-inside 3.4e-14  Visible-distance 4.5e-15
  U-max-coplanar 4.5e-15  Width-outside 8.9e-15  _wide-facet 2.7e-14
  _maxoutside 8.9e-15


#### Errors

Apparently for 12 point clouds no normals could be calculated:
DONE: point cloud yes
DONE: normals yes

- '../data/pc_cad/0063/00637471.ply' DONE DONE FJANCDJ
- '../data/pc_cad/0074/00745094.ply' DONE DONE spdivnsdkvon
- '../data/pc_cad/0064/00642362.ply' DONE DONE sdpoivasdfoivn
- '../data/pc_cad/0042/00426282.ply' -> Here is a data problem! When transforming the cad object to a point cloud the warning "x faces have been skipped due to null triangulation" appears. In this case the CAD object has only two faces (it is a pipe), therefore the point cloud is only a line. DONE sdoivnaodivn
- '../data/pc_cad/0061/00616862.ply' DONE DONE onoj noj n
- '../data/pc_cad/0064/00642363.ply' DONE DONE soidvnos
- '../data/pc_cad/0054/00547409.ply' DONE DONE sdfws
- '../data/pc_cad/0057/00577907.ply' DONE DONE oergnvso
- '../data/pc_cad/0045/00453085.ply' DONE DONE sdfpvksnd
- '../data/pc_cad/0009/00090870.ply' DONE DONE wc w
- '../data/pc_cad/0005/00056341.ply' DONE DONE sdokvedl
- '../data/pc_cad/0083/00830386.ply' DONE DONE

The issue with these point clouds is that they are planes with an extent in the z direction of zero or close to zero. Therefore we have to manually add the normals.

In [149]:
pc_path ='../data/pc_cad/0083/00830386.ply'
axis = 0

new_pc_path = pc_path.replace('pc_cad','pc_cad_norm')
print(new_pc_path)
point_cloud = o3d.io.read_point_cloud(pc_path)
print(point_cloud)
points = np.asarray(point_cloud.points)
normals = np.zeros((8096, 3))
normals[:, axis] = 1
pc_n = np.hstack((points, normals))
pcn = o3d.geometry.PointCloud()
pcn.points = o3d.utility.Vector3dVector(points)
pcn.normals = o3d.utility.Vector3dVector(normals)
#o3d.visualization.draw_geometries([pcn], point_show_normal=True)

../data/pc_cad_norm/0083/00830386.ply
PointCloud with 8096 points.


In [151]:
norms = np.linalg.norm(pc_n[:, 3:], axis=1)
print(np.unique(norms))

[1.]


In [152]:
o3d.io.write_point_cloud(new_pc_path, pcn)

True

In [159]:
len(train)

160982

In [9]:
train_norm = PointCloudEmbeddingSequenceDataset("../data", "train", use_normals = True)
val_norm = PointCloudEmbeddingSequenceDataset("../data", "validation", use_normals = True)
test_norm = PointCloudEmbeddingSequenceDataset("../data", "test", use_normals = True)

In [14]:
for dataset in [train, val, test]:
    for i, data in enumerate(dataset):
        print(i, end = '\r')
        pc = data['pc']
        assert pc.shape==(2048, 3), data['id']

KeyboardInterrupt: 

In [6]:
import random
random_ids = np.random.randint(0, len(train_norm), size=100).tolist()
for id in random_ids:
    pc_path = train_norm.get_pc_path(id)
    point_cloud = o3d.io.read_point_cloud(pc_path)
    o3d.visualization.draw_geometries([point_cloud], point_show_normal=True)



[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARN

In [153]:
path = '../data/pc_cad_norm/0083/00830386.ply'
point_cloud = o3d.io.read_point_cloud(path)
o3d.visualization.draw_geometries([point_cloud], point_show_normal=True)

[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display


In [105]:
point_cloud.normals

std::vector<Eigen::Vector3d> with 0 elements.
Use numpy.asarray() to access data.

In [98]:
norms = np.linalg.norm(point_cloud.normals, axis=1)
print(np.unique(norms))

[0.99999992 0.99999992 0.99999993 ... 1.00000007 1.00000007 1.00000008]


In [71]:
dataset = PointCloudEmbeddingSequenceDataset("../data", 'train')
pcs = dataset.pc
print(f"Dataset contains {len(dataset)} samples.")

find_index("00745094", dataset)

Dataset contains 160982 samples.
8177


In [43]:
RECORD_FILE = os.path.join("../data", "train_val_test_split.json")
with open(RECORD_FILE, "r") as fp:
    all_data = json.load(fp)

In [57]:
print(len(all_data['train']))
for i,path in enumerate(all_data['train']):
    print(path)
    break
  #  if path == '0042/00426282':
   #     print(i)

161240
0067/00675619


In [12]:
lol = np.zeros((8096,3))
print(lol.shape)

(8096, 3)


In [36]:
pcd = o3d.io.read_point_cloud("../data/pc_cad_norm/0067/00675619.ply")

In [39]:
print(pcd)
o3d.visualization.draw_geometries([pcd], point_show_normal=True)

PointCloud with 8096 points.
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display


In [6]:
points = tensor.numpy()  
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)
pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamKNN(knn=30))
pcd.orient_normals_consistent_tangent_plane(50)
normals = np.asarray(pcd.normals)
point_cloud_with_normals = np.hstack((points, normals))
tensor_with_normals = torch.tensor(point_cloud_with_normals)
o3d.visualization.draw_geometries([pcd], point_show_normal=True)


TypeError: estimate_normals(): incompatible function arguments. The following argument types are supported:
    1. (self: open3d.cpu.pybind.geometry.PointCloud, search_param: open3d.cpu.pybind.geometry.KDTreeSearchParam = KDTreeSearchParamKNN with knn = 30, fast_normal_computation: bool = True) -> None

Invoked with: PointCloud with 2048 points.; kwargs: search_param=30

In [44]:
normals.shape

(2048, 3)

In [49]:
np.linalg.norm(normals, axis=1)

array([1., 1., 1., ..., 1., 1., 1.])

In [102]:
import os
import shutil

def copy_to_pc_cad_norm(file_paths):
    for path in file_paths:
        norm_path = path.replace("pc_cad", "pc_cad_norm")
        os.makedirs(os.path.dirname(norm_path), exist_ok=True)
        shutil.copy(path, norm_path)
        print(f"Copied {path} → {norm_path}")
copy_to_pc_cad_norm(['../data/pc_cad/0063/00637471.ply', '../data/pc_cad/0074/00745094.ply', '../data/pc_cad/0064/00642362.ply', '../data/pc_cad/0042/00426282.ply', '../data/pc_cad/0061/00616862.ply', '../data/pc_cad/0064/00642363.ply', '../data/pc_cad/0054/00547409.ply', '../data/pc_cad/0057/00577907.ply', '../data/pc_cad/0045/00453085.ply', '../data/pc_cad/0009/00090870.ply', '../data/pc_cad/0005/00056341.ply', '../data/pc_cad/0083/00830386.ply'])

Copied ../data/pc_cad/0063/00637471.ply → ../data/pc_cad_norm/0063/00637471.ply
Copied ../data/pc_cad/0074/00745094.ply → ../data/pc_cad_norm/0074/00745094.ply
Copied ../data/pc_cad/0064/00642362.ply → ../data/pc_cad_norm/0064/00642362.ply
Copied ../data/pc_cad/0042/00426282.ply → ../data/pc_cad_norm/0042/00426282.ply
Copied ../data/pc_cad/0061/00616862.ply → ../data/pc_cad_norm/0061/00616862.ply
Copied ../data/pc_cad/0064/00642363.ply → ../data/pc_cad_norm/0064/00642363.ply
Copied ../data/pc_cad/0054/00547409.ply → ../data/pc_cad_norm/0054/00547409.ply
Copied ../data/pc_cad/0057/00577907.ply → ../data/pc_cad_norm/0057/00577907.ply
Copied ../data/pc_cad/0045/00453085.ply → ../data/pc_cad_norm/0045/00453085.ply
Copied ../data/pc_cad/0009/00090870.ply → ../data/pc_cad_norm/0009/00090870.ply
Copied ../data/pc_cad/0005/00056341.ply → ../data/pc_cad_norm/0005/00056341.ply
Copied ../data/pc_cad/0083/00830386.ply → ../data/pc_cad_norm/0083/00830386.ply
